In [6]:
#req libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

In [8]:
df=pd.read_csv("/content/banking_loan_default_dataset_500_rows.csv")
df.head()

,customer_id,age,income_monthly,loan_amount,credit_score,employment_years,debt_to_income,existing_loans,dependents,bank_balance,interest_rate,employment_status,marital_status,education,loan_purpose,city_tier,previous_defaults,loan_default
0,CUST1362,29,143036.0,200322,574.0,4.7,0.22,5,4,301629,13.12,Business,Married,Graduate,Home,Tier 3,0,No
1,CUST1074,28,56102.0,863738,810.0,2.8,0.13,5,3,620324,8.92,Salaried,Married,Undergraduate,Home,Tier 2,0,Yes
2,CUST1375,40,93703.0,428165,687.0,8.3,0.51,3,2,94037,10.44,Business,Divorced,Undergraduate,Education,Tier 3,0,Yes
3,CUST1156,52,123787.0,364997,651.0,0.6,0.17,0,0,483454,8.08,Self-employed,Married,Postgraduate,Education,Tier 3,1,Yes
4,CUST1105,25,48355.0,855688,662.0,3.6,0.47,3,0,906311,8.91,Salaried,Married,Undergraduate,Education,Tier 1,0,Yes


In [10]:
df.shape
df.info()
df.isnull().sum().sort_values(ascending=False).head(15)#top 15

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        500 non-null    object 
 1   age                500 non-null    int64  
 2   income_monthly     485 non-null    float64
 3   loan_amount        500 non-null    int64  
 4   credit_score       485 non-null    float64
 5   employment_years   485 non-null    float64
 6   debt_to_income     485 non-null    float64
 7   existing_loans     500 non-null    int64  
 8   dependents         500 non-null    int64  
 9   bank_balance       500 non-null    int64  
 10  interest_rate      500 non-null    float64
 11  employment_status  500 non-null    object 
 12  marital_status     500 non-null    object 
 13  education          485 non-null    object 
 14  loan_purpose       500 non-null    object 
 15  city_tier          500 non-null    object 
 16  previous_defaults  500 non

,0
credit_score,15
income_monthly,15
education,15
debt_to_income,15
employment_years,15
customer_id,0
age,0
loan_amount,0
dependents,0
existing_loans,0


In [11]:
target_column='loan_default'
df=df.dropna(subset=[target_column]).copy()
X=df.drop(columns=[target_column])
y=df[target_column]
print('Target values:')
print(y.value_counts())

Target values:
loan_default
Yes    357
No     143
Name: count, dtype: int64


In [12]:
label_encoder=LabelEncoder()
y=label_encoder.fit_transform(y.astype(str))

print('Classes:', list(label_encoder.classes_))
print('Encoded classes:',np.unique(y))

Classes: ['No', 'Yes']
Encoded classes: [0 1]


In [13]:
numeric_features=X.select_dtypes(include=np.number).columns.tolist()
categorical_features=X.select_dtypes(exclude=np.number).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Numeric features: ['age', 'income_monthly', 'loan_amount', 'credit_score', 'employment_years', 'debt_to_income', 'existing_loans', 'dependents', 'bank_balance', 'interest_rate', 'previous_defaults']
Categorical features: ['customer_id', 'employment_status', 'marital_status', 'education', 'loan_purpose', 'city_tier']
